3

In [3]:
import yaml

with open("config.yaml", "r") as file:
    config = yaml.safe_load(file)

print(config)

{'paths': {'input_dir': 'dataset', 'output_dir': 'cleaned_data', 'log_dir': 'logs', 'report_dir': 'reports', 'manifest_file': 'manifest.json'}, 'settings': {'null_threshold': 0.5, 'remove_duplicates': True, 'output_formats': ['csv', 'json']}, 'mappings': {'order_status': {'delivered': 'completed', 'shipped': 'in_transit', 'canceled': 'cancelled'}}}


In [4]:
print(config["paths"]["input_dir"])
print(config["paths"]["output_dir"])
print(config["settings"]["null_threshold"])


dataset
cleaned_data
0.5


1

In [6]:
import logging
import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True
)

logging.info("Data pipeline started")


2026-11-23 14:26:37,719 - INFO - Data pipeline started


In [7]:
df = pd.read_csv("dataset/olist_orders_dataset.csv")

logging.info("CSV file loaded successfully")
logging.info("Total records: %d", len(df))

2026-11-23 14:26:46,078 - INFO - CSV file loaded successfully
2026-11-23 14:26:46,079 - INFO - Total records: 99441


In [8]:
null_counts = df.isnull().sum()

if null_counts.sum() > 0:
    logging.warning(
        "Missing values found: %d",
        null_counts.sum()
    )
else:
    logging.info("No missing values found")

2026-11-23 14:26:55,609 - WARNING - Missing values found: 4908


In [9]:
duplicate_count = df.duplicated().sum()

if duplicate_count > 0:
    logging.warning(
        "Duplicate records found: %d",
        duplicate_count
    )
else:
    logging.info("No duplicate records found")

2026-11-23 14:27:10,914 - INFO - No duplicate records found


In [10]:
try:
    pd.read_csv("dataset/invalid_file.csv")

except FileNotFoundError:
    logging.error("Requested CSV file was not found")

2026-11-23 14:27:22,914 - ERROR - Requested CSV file was not found


2

In [11]:
from pathlib import Path
from datetime import datetime
import logging

# Create logs folder
log_dir = Path(config["paths"]["log_dir"])
log_dir.mkdir(parents=True, exist_ok=True)

# Generate timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Create log filename
log_file = log_dir / f"pipeline_{timestamp}.log"

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(log_file, encoding="utf-8"),
        logging.StreamHandler()
    ],
    force=True
)

logging.info("Pipeline execution started")

print("Log file created:", log_file)

2026-11-23 14:29:20,339 - INFO - Pipeline execution started


Log file created: logs\pipeline_20261123_142920.log


In [12]:
logging.info("CSV file loaded successfully")
logging.warning("Missing values found")
logging.error("Sample error message")

2026-11-23 14:29:36,308 - INFO - CSV file loaded successfully
2026-11-23 14:29:36,310 - WARNING - Missing values found
2026-11-23 14:29:36,313 - ERROR - Sample error message


In [13]:
list(log_dir.glob("*.log"))

[WindowsPath('logs/pipeline_20261123_142920.log')]

In [14]:
print(log_file.read_text(encoding="utf-8"))

2026-11-23 14:29:20,339 - INFO - Pipeline execution started
2026-11-23 14:29:36,308 - INFO - CSV file loaded successfully
2026-11-23 14:29:36,310 - WARNING - Missing values found
2026-11-23 14:29:36,313 - ERROR - Sample error message



4

In [15]:
import pandas as pd

df = pd.read_csv("dataset/olist_orders_dataset.csv")

print("Original records:", len(df))

Original records: 99441


In [16]:
cleaned_df = df.copy()

# Remove duplicates
if config["settings"]["remove_duplicates"]:
    cleaned_df = cleaned_df.drop_duplicates()

# Standardize column names
cleaned_df.columns = (
    cleaned_df.columns.str.strip().str.lower()
)

print("Cleaned records:", len(cleaned_df))

Cleaned records: 99441


In [17]:
output_dir = Path(config["paths"]["output_dir"])
output_dir.mkdir(parents=True, exist_ok=True)

cleaned_df.to_csv(
    output_dir / "orders_cleaned.csv",
    index=False
)

In [18]:
cleaned_df.to_json(
    output_dir / "orders_cleaned.json",
    orient="records",
    indent=4
)

In [20]:
print("CSV exists:",
      (output_dir / "orders_cleaned.csv").exists())

print("JSON exists:",
      (output_dir / "orders_cleaned.json").exists())

CSV exists: True
JSON exists: True


5

In [22]:
import time
import json

# Record pipeline start time
start_time = time.time()

print("Pipeline started")

# Calculate pipeline duration
end_time = time.time()
duration = end_time - start_time

# Create run summary
summary = {
    "total_records": len(df),
    "duration_seconds": round(duration, 2),
    "status": "Success"
}

# Save summary as JSON
with open("run_summary.json", "w") as file:
    json.dump(summary, file, indent=4)

print("Run summary JSON created successfully")
print(summary)

Pipeline started
Run summary JSON created successfully
{'total_records': 99441, 'duration_seconds': 0.0, 'status': 'Success'}


6

In [39]:

import os
import json
import pandas as pd

# Step 1: Define file paths
input_file = "dataset\\olist_orders_dataset.csv"  # Replace with your actual CSV filename
manifest_file = "manifest.json"

# Step 2: Read existing manifest
if os.path.exists(manifest_file):
    try:
        with open(manifest_file, "r") as file:
            manifest = json.load(file)

        if not isinstance(manifest, dict) or not isinstance(
            manifest.get("processed_files"), list
        ):
            manifest = {"processed_files": []}

    except json.JSONDecodeError:
        manifest = {"processed_files": []}
else:
    manifest = {"processed_files": []}

# Step 3: Check whether file was already processed
if input_file in manifest["processed_files"]:

    print("File already processed. Skipping:", input_file)

else:

    print("New file detected. Processing:", input_file)

    # Step 4: Load CSV
    df = pd.read_csv(input_file)

    print("File loaded successfully")
    print("Total records:", len(df))

    # Step 5: Perform your existing validation and cleaning here
    # Add your actual cleaning code before updating the manifest.

    # Step 6: Save clean output
    df.to_csv("cleaned_output.csv", index=False)

    # Step 7: Update manifest after successful processing
    manifest["processed_files"].append(input_file)

    with open(manifest_file, "w") as file:
        json.dump(manifest, file, indent=4)

    print("File processed successfully")
    print("Manifest updated successfully")


File already processed. Skipping: dataset\olist_orders_dataset.csv


7

In [40]:
import yaml

with open("config.yaml", "r") as file:
    config = yaml.safe_load(file)

print(config)

{'paths': {'input_dir': 'dataset', 'output_dir': 'cleaned_data', 'log_dir': 'logs', 'report_dir': 'reports', 'manifest_file': 'manifest.json'}, 'settings': {'null_threshold': 0.5, 'remove_duplicates': True, 'output_formats': ['csv', 'json']}, 'mappings': {'order_status': {'delivered': 'completed', 'shipped': 'in_transit', 'canceled': 'cancelled'}}}


In [41]:
import os

# Read paths from configuration
input_dir = config["paths"]["input_dir"]
output_dir = config["paths"]["output_dir"]
log_dir = config["paths"]["log_dir"]
report_dir = config["paths"]["report_dir"]

# Create directories if they don't exist
os.makedirs(output_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)
os.makedirs(report_dir, exist_ok=True)

print("All pipeline directories are ready")
print("Input directory:", input_dir)
print("Output directory:", output_dir)
print("Log directory:", log_dir)
print("Report directory:", report_dir)

All pipeline directories are ready
Input directory: dataset
Output directory: cleaned_data
Log directory: logs
Report directory: reports


In [42]:
import importlib

import reader
import validator
import transformer

# Reload modules to get the latest changes
importlib.reload(reader)
importlib.reload(validator)
importlib.reload(transformer)

print("All Python modules imported successfully")

All Python modules imported successfully


In [43]:
import logging
from datetime import datetime
import os

# Generate timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Create log filename
log_file = os.path.join(
    config["paths"]["log_dir"],
    f"pipeline_{timestamp}.log"
)

# Configure logging
logging.basicConfig(
    filename=log_file,
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True
)

# Test logging
logging.info("Pipeline logging configured successfully")

print("Logging configured successfully")
print("Log file:", log_file)

Logging configured successfully
Log file: logs\pipeline_20261124_103431.log


In [44]:
import os

# Get input directory from config
input_dir = config["paths"]["input_dir"]

# Define CSV file path
input_file = os.path.join(
    input_dir,
    "olist_orders_dataset.csv"
)

# Load CSV using reader.py
df = reader.read_csv_file(input_file)

# Log the result
logging.info("CSV file loaded successfully")

print("CSV file loaded successfully")
print("Total records:", len(df))
print("Columns:", df.columns.tolist())

CSV file loaded successfully
Total records: 99441
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


In [45]:
# Check missing values
null_result = validator.check_nulls(df)

# Check duplicate records
duplicate_result = validator.check_duplicates(df)

# Log validation results
logging.info("Null validation completed")
logging.info("Duplicate validation completed")

print("Null validation completed")
print(null_result)

print("\nDuplicate validation completed")
print(duplicate_result)

Null validation completed
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Duplicate validation completed
0


In [46]:
# Step 1: Standardize column names
cleaned_df = transformer.standardize_columns(df)

# Step 2: Remove duplicate records
cleaned_df = transformer.remove_duplicates(cleaned_df)

# Step 3: Apply column mapping from config.yaml
column_mapping = config["mappings"].get("columns", {})

if column_mapping:
    cleaned_df = transformer.apply_column_mapping(
        cleaned_df,
        column_mapping
    )

# Log cleaning results
logging.info("Column names standardized")
logging.info("Duplicate records removed")

print("Data cleaning completed successfully")
print("Original records:", len(df))
print("Cleaned records:", len(cleaned_df))
print("Cleaned columns:", cleaned_df.columns.tolist())

Data cleaning completed successfully
Original records: 99441
Cleaned records: 99441
Cleaned columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


In [47]:
# Get mapping from config.yaml
status_mapping = config["mappings"]["order_status"]

# Apply mapping
cleaned_df["order_status"] = (
    cleaned_df["order_status"].replace(status_mapping)
)

# Log the operation
logging.info("Order status mapping applied successfully")

print("Order status mapping completed")
print(cleaned_df["order_status"].value_counts())

Order status mapping completed
order_status
completed      96478
in_transit      1107
cancelled        625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [48]:
import os
import json

# Get output directory and formats from config
output_dir = config["paths"]["output_dir"]
output_formats = config["settings"]["output_formats"]

# Save CSV
if "csv" in output_formats:
    csv_path = os.path.join(
        output_dir, "cleaned_orders.csv"
    )

    cleaned_df.to_csv(csv_path, index=False)
    logging.info("Cleaned CSV file saved successfully")
    print("CSV file saved:", csv_path)

# Save JSON
if "json" in output_formats:
    json_path = os.path.join(
        output_dir, "cleaned_orders.json"
    )

    cleaned_df.to_json(
        json_path,
        orient="records",
        indent=4,
        date_format="iso"
    )

    logging.info("Cleaned JSON file saved successfully")
    print("JSON file saved:", json_path)

print("All configured output files saved successfully")

CSV file saved: cleaned_data\cleaned_orders.csv
JSON file saved: cleaned_data\cleaned_orders.json
All configured output files saved successfully


final 6 

In [49]:
import os
import json
import time
import yaml
import pandas as pd
import logging
import reader
import validator
import transformer

def run_pipeline():

    start_time = time.time()

    # 1. Read configuration
    with open("config.yaml", "r") as file:
        config = yaml.safe_load(file)

    print("Configuration loaded")

    # 2. Read CSV
    input_file = os.path.join(
        config["paths"]["input_dir"],
        "olist_orders_dataset.csv"
    )

    df = reader.read_csv_file(input_file)

    print("CSV loaded:", len(df))

    # 3. Validate data
    null_result = validator.check_nulls(df)
    duplicate_result = validator.check_duplicates(df)

    print("Data validation completed")

    # 4. Clean data
    cleaned_df = transformer.standardize_columns(df)
    cleaned_df = transformer.remove_duplicates(cleaned_df)

    # 5. Apply status mapping
    status_mapping = config["mappings"]["order_status"]

    cleaned_df["order_status"] = (
        cleaned_df["order_status"].replace(status_mapping)
    )

    print("Data cleaning completed")

    # 6. Save CSV and JSON
    output_dir = config["paths"]["output_dir"]
    os.makedirs(output_dir, exist_ok=True)

    cleaned_df.to_csv(
        os.path.join(output_dir, "cleaned_orders.csv"),
        index=False
    )

    cleaned_df.to_json(
        os.path.join(output_dir, "cleaned_orders.json"),
        orient="records",
        indent=4
    )

    print("CSV and JSON files saved")

    # 7. Calculate duration
    duration = round(time.time() - start_time, 2)

    # 8. Create run summary
    summary = {
        "total_records": len(df),
        "valid_records": len(cleaned_df),
        "rejected_records": len(df) - len(cleaned_df),
        "duration_seconds": duration,
        "status": "Success"
    }

    report_dir = config["paths"]["report_dir"]
    os.makedirs(report_dir, exist_ok=True)

    with open(os.path.join(report_dir, "run_summary.json"), "w") as file:
        json.dump(summary, file, indent=4)

    print("Run summary created")

    print("\nPipeline completed successfully")
    print(summary)


# Execute the entire pipeline
run_pipeline()

Configuration loaded
CSV loaded: 99441
Data validation completed
Data cleaning completed
CSV and JSON files saved
Run summary created

Pipeline completed successfully
{'total_records': 99441, 'valid_records': 99441, 'rejected_records': 0, 'duration_seconds': 0.9, 'status': 'Success'}


7

In [51]:
%pip install ruff

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [52]:
!ruff --version

ruff 0.17.6


In [54]:
import csv
import pandas as pd

In [55]:
import json

In [56]:
!ruff check --fix reader.py validator.py transformer.py

Found 2 errors (2 fixed, 0 remaining).


In [57]:
!ruff format reader.py validator.py transformer.py

3 files reformatted


In [ ]:
!ruff check reader.py validator.py transformer.py



All checks passed!
